# Step 1: TPU 기반 KLUE-RoBERTa 워크플로우 분류 모델 학습

이 노트북은 **Google Colab TPU**를 사용하여 워크플로우 요청과 일반 작업 요청을 분류하는 모델을 학습합니다.

## 주요 변경 사항 (시나리오 2 및 오류 수정 반영)
- **채용 워크플로우 데이터 포함**: 약 300개의 채용 관련 샘플이 추가된 최신 데이터셋 사용
- **TPU 호환성 수정**: `fused=True` 관련 런타임 에러 방지를 위해 옵티마이저 설정 변경
- **최신 라이브러리 대응**: `evaluation_strategy`를 `eval_strategy`로 수정

## 1. 환경 설정 및 라이브러리 설치

In [ ]:
# TPU 관련 라이브러리 및 필수 패키지 설치
!pip install -q transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn

# Colab TPU 환경을 위한 PyTorch XLA 설치 (필요시)
# !pip install cloud-tpu-client==0.10 torch==2.0.0 torchvision==0.15.1 https://storage.googleapis.com/tpu-pytorch/wheels/colab/torch_xla-2.0-cp310-cp310-linux_x86_64.whl

In [ ]:
import os
import json
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from pathlib import Path

from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments, 
    EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# TPU 체크 및 디바이스 설정
use_tpu = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    import torch_xla.core.xla_model as xm
    # 실제로 TPU 디바이스를 가져올 수 있는지 확인
    tpu_device = xm.xla_device()
    if tpu_device:
        device = tpu_device
        use_tpu = True
        print(f"✅ TPU 감지됨: {device}")
except Exception as e:
    print(f"⚠️ TPU를 사용할 수 없습니다 ({type(e).__name__}). {device}를 사용합니다.")
    use_tpu = False

## 2. 데이터셋 로드

**[중요]** 로컬의 `data/step1_classification_dataset/` 폴더에 있는 `train.csv`, `val.csv`, `test.csv` 파일을 코랩에 업로드하세요.

In [ ]:
# 파일 업로드 확인 및 로드
data_dir = "./" # 파일이 현재 루트에 업로드되었다고 가정

def load_data(filename):
    path = os.path.join(data_dir, filename)
    if os.path.exists(path):
        return pd.read_csv(path)
    else:
        print(f"❌ {filename} 파일을 찾을 수 없습니다. 업로드를 확인해주세요.")
        return None

train_df = load_data("train.csv")
val_df = load_data("val.csv")
test_df = load_data("test.csv")

if train_df is not None:
    print(f"📊 데이터 로드 완료")
    print(f"- Train: {len(train_df)}")
    print(f"- Val: {len(val_df)}")
    print(f"- Test: {len(test_df)}")
    
    # 클래스 분포 확인
    sns.countplot(x='label', data=train_df)
    plt.title('Class Distribution (0: General, 1: Workflow)')
    plt.show()

## 3. 전처리 및 모델 준비

In [ ]:
model_name = "klue/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["query"], truncation=True, padding="max_length", max_length=128)

# Dataset 변환
train_dataset = Dataset.from_pandas(train_df[['query', 'label']]).map(tokenize_function, batched=True)
val_dataset = Dataset.from_pandas(val_df[['query', 'label']]).map(tokenize_function, batched=True)
test_dataset = Dataset.from_pandas(test_df[['query', 'label']]).map(tokenize_function, batched=True)

# 모델 로드
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

## 4. 학습 설정 및 실행

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=32 if use_tpu else 16,
    per_device_eval_batch_size=32 if use_tpu else 16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    # TPU 설정 최적화 및 오류 수정
    tpu_num_cores=8 if use_tpu else None,
    dataloader_num_workers=4,
    report_to="none",
    # ✅ TPU에서는 fused optimizer를 지원하지 않으므로 표준 adamw_torch 사용
    optim="adamw_torch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("🚀 학습 시작...")
trainer.train()

## 5. 성능 평가 및 시각화

In [ ]:
print("🧐 테스트 세트 평가 중...")
results = trainer.evaluate(test_dataset)
print(f"✅ 최종 결과: {results}")

# 혼동 행렬 시각화
preds = trainer.predict(test_dataset)
pred_labels = np.argmax(preds.predictions, axis=-1)
true_labels = test_dataset["label"]

cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['General', 'Workflow'], 
            yticklabels=['General', 'Workflow'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

## 6. 모델 저장 및 다운로드

In [ ]:
save_path = "./workflow_classifier_final"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

# 압축하여 다운로드
!zip -r model.zip {save_path}

from google.colab import files
files.download("model.zip")